# Notebook 01 — EDA & Data Preparation

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** A — Data & EDA (Section 2 below) + C — MLOps / Data Pipeline (everything else)  
**Estimated Runtime:** 2-5 minutes

---

## What this notebook does

1. Loads the crypto scam dataset and uploads the raw file to S3
2. Exploratory Data Analysis — **placeholder, Royston's section (Focus A)**
3. Runs the shared data pipeline (`utils/preprocessing.py`): text cleaning, engineered indicator features, TF-IDF, stratified train/test split
4. Saves the processed train/test data and the fitted TF-IDF vectorizer to S3 for Notebook 02/03

## Dataset

- Local copy: `data/crypto_scam_dataset.csv` (already in this repo)
- Source: [Kaggle — Crypto Scam Dataset](https://www.kaggle.com/datasets/theeyanyan/crypto-scam-dataset/data), as cited in the project proposal
- 10,000 rows, columns `id`, `platform`, `text`, `label` (`scam` / `legit`), no missing values

> **Adapted for team03 (Ong Hui Lin, Student 2 — MLOps & Deployment) from the ITI113 course template notebook `01_eda_and_data_preparation.ipynb`.** The tutor's notebook mixes two things: EDA/insights (individually graded as Focus A — Royston's) and data loading/preparation (the MLOps data pipeline — this project's job). **Section 2 (EDA) is intentionally left as a placeholder for Royston to fill in** rather than written on his behalf. Everything else follows the tutor's structure as closely as the data allows, adapted to text data (TF-IDF) instead of tabular clinical features, and reuses `utils/preprocessing.py` rather than reimplementing cleaning/feature logic inline, so the notebook and the Streamlit app stay in sync — sections marked **`# ADAPTED`** explain what had to change.

In [ ]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade boto3 botocore sagemaker

## 0. Configuration

Edit the values in this cell. Everything else references these variables.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # so `utils.*` imports resolve from the repo root

import sagemaker, boto3

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

TEAM_ID = "team03"
STUDENT_ID = "s301"

BUCKET = "nyp-26s1-iti113"
PROJECT_NAME = "crypto-scam-detector"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"  # matches the PREFIX pattern used in Notebook 02

RANDOM_STATE = 42   # matches utils.preprocessing.run_pipeline()'s default
TEST_SIZE    = 0.20 # matches utils.preprocessing.run_pipeline()'s default

print(f"Bucket     : {BUCKET}")
print(f"Prefix     : {PREFIX}")
print(f"Region     : {region}")
print(f"Role       : {role.split('/')[-1]}")
print(f"Team ID    : {TEAM_ID}")
print(f"Student ID : {STUDENT_ID}")

## 1. Load Dataset

**ADAPTED:** the tutor's version downloads the UCI Heart Disease dataset from a public URL. This project's dataset already lives locally at `data/crypto_scam_dataset.csv` (downloaded from Kaggle per the proposal), so this loads it via `utils.preprocessing.load_raw_data()` — the same loader the shared pipeline and the Streamlit app use — rather than re-fetching it.

| Column | Description |
|--------|-------------|
| id | Unique message identifier |
| platform | Source platform (Telegram, X/Twitter, SMS, Email, Discord, Reddit) |
| text | Message content |
| label | `scam` or `legit` |

In [ ]:
from utils.preprocessing import load_raw_data

df_raw = load_raw_data()

print(f'Shape : {df_raw.shape}')
print(f'Label : {df_raw["label"].value_counts().to_dict()}')
df_raw.head()

In [ ]:
# Upload raw dataset to S3 for version control
import io

s3 = boto3.client('s3')
buf = io.StringIO()
df_raw.to_csv(buf, index=False)
s3.put_object(Bucket=BUCKET, Key=f'{PREFIX}/raw/crypto_scam_dataset.csv', Body=buf.getvalue())
RAW_S3_URI = f's3://{BUCKET}/{PREFIX}/raw/crypto_scam_dataset.csv'
print(f'Raw data uploaded to: {RAW_S3_URI}')

## 2. Exploratory Data Analysis

**Placeholder — Royston's section (Focus A).** The Progress Check rubric wants completed EDA with insights and patterns identified here: missing-value checks (none, per the load above, but worth confirming), class balance (`scam` vs `legit`), platform distribution, message length distribution, common scam-language patterns, and any data-quality issues or bias observations for the AI Governance checklist. Left empty intentionally rather than filled in on his behalf — see `df_raw` from Section 1 above as the starting point.

## 3. Run the Shared Data Pipeline

**ADAPTED:** the tutor's version does feature engineering (`age_group`, `high_risk_count`), train/test split, and `StandardScaler` inline, specific to the tabular heart-disease features. This project's equivalent logic (text cleaning, the proposal's engineered indicator features, TF-IDF fitting, stratified train/test split) already lives in `utils/preprocessing.py` — reused here rather than duplicated, so this notebook, Notebook 02, and the Streamlit app all stay in sync. No feature scaling is applied, matching what the rest of the project already does (TF-IDF features are not scaled elsewhere either).

In [ ]:
from utils.preprocessing import run_pipeline

result = run_pipeline(test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_df, test_df, vectorizer = result["train_df"], result["test_df"], result["vectorizer"]

print(f'Train: {train_df.shape[0]} rows  |  Test: {test_df.shape[0]} rows')
print(f'Train label balance: {(train_df["label"]=="scam").mean():.3f}  '
      f'(full dataset: {(df_raw["label"]=="scam").mean():.3f})')
print(f'Test  label balance: {(test_df["label"]=="scam").mean():.3f}  (stratification confirmed)')
print(f'TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}')

## 4. Save Processed Data to S3

In [ ]:
# ADAPTED: the tutor's version saves four numeric CSVs (train/test features and labels
# split apart), which fits tabular data. This project's processed output is train_df /
# test_df (text + engineered features + label together, matching data/processed/*.csv
# already produced locally) plus the fitted TF-IDF vectorizer -- so three artifacts are
# uploaded instead. If Notebook 02/03 are later switched to load from S3 instead of
# re-running the pipeline locally, they should read these three, not the tutor's four.

import joblib
import tempfile

def save_csv_to_s3(df, bucket, key):
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    return f's3://{bucket}/{key}'

def save_joblib_to_s3(obj, bucket, key):
    with tempfile.NamedTemporaryFile(suffix='.joblib') as tmp:
        joblib.dump(obj, tmp.name)
        s3.upload_file(tmp.name, bucket, key)
    return f's3://{bucket}/{key}'

p = f'{PREFIX}/processed'
paths = {
    'train'            : save_csv_to_s3(train_df, BUCKET, f'{p}/train.csv'),
    'test'             : save_csv_to_s3(test_df, BUCKET, f'{p}/test.csv'),
    'tfidf_vectorizer' : save_joblib_to_s3(vectorizer, BUCKET, f'{p}/tfidf_vectorizer.joblib'),
}
for name, uri in paths.items():
    print(f'{name:<20}: {uri}')

PROCESSED_PREFIX = f's3://{BUCKET}/{p}'
print(f'\nProcessed prefix: {PROCESSED_PREFIX}')
print('Next: open Notebook 02 to run baseline experiments with MLflow.')

---
## Checklist before Notebook 02

- [ ] Raw dataset uploaded to S3
- [ ] EDA completed by Royston (Section 2) — insights, class balance, platform distribution, data quality / bias observations for AI Governance
- [ ] Shared data pipeline (`utils/preprocessing.py`) run successfully — clean text, engineered indicator features, TF-IDF fitted, stratified train/test split
- [ ] Processed train/test data and TF-IDF vectorizer saved to S3
- [ ] EDA observations and governance flags documented for the checklist